# Per-start dataset (2021–2026)

One row per starting pitcher appearance. Targets are the starter's hits, strikeouts, walks and earned runs in that game. Features only use games from earlier dates.

Pulls box scores from the MLB Stats API (`statsapi.mlb.com`), saves gzipped raw CSVs to `data/raw/`, then builds `data/starts_dataset.csv.gz`.

Column prefixes: `p_` pitcher form, `lu_` lineup averages, `b1_`–`b9_` each opposing batter by slot (for the lineup-aware model).

In [1]:
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import requests

API = "https://statsapi.mlb.com/api/v1"
SEASONS = range(2021, 2027)
TRAIN_SEASONS = [2021, 2022, 2023, 2024]
DATA = Path("data")
RAW = DATA / "raw"
RAW.mkdir(parents=True, exist_ok=True)

session = requests.Session()

## Pull box scores

In [2]:
def get(url, params=None, tries=5):
    for i in range(tries):
        try:
            r = session.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if i == tries - 1:
                raise
            time.sleep(2 ** i)


def schedule(season):
    data = get(f"{API}/schedule", {"sportId": 1, "season": season, "gameType": "R"})
    games = {}
    for d in data["dates"]:
        for g in d["games"]:
            if g["status"]["codedGameState"] == "F":
                games[g["gamePk"]] = d["date"]
    return games


def parse_game(game_pk, date):
    box = get(f"{API}/game/{game_pk}/boxscore")
    starts, bats = [], []
    for side, opp in [("home", "away"), ("away", "home")]:
        team = box["teams"][side]
        base = {"game_pk": game_pk, "date": date, "team_id": team["team"]["id"],
                "opp_id": box["teams"][opp]["team"]["id"], "home": int(side == "home")}

        # first pitcher listed is the starter
        if team["pitchers"]:
            p = team["players"][f"ID{team['pitchers'][0]}"]
            s = p["stats"]["pitching"]
            starts.append({**base, "pitcher_id": p["person"]["id"],
                           "outs": s.get("outs", 0), "bf": s.get("battersFaced", 0),
                           "pitches": s.get("numberOfPitches", 0),
                           "h": s.get("hits", 0), "k": s.get("strikeOuts", 0),
                           "bb": s.get("baseOnBalls", 0), "er": s.get("earnedRuns", 0),
                           "hr": s.get("homeRuns", 0), "hbp": s.get("hitByPitch", 0)})

        for p in team["players"].values():
            if "battingOrder" not in p:
                continue
            s = p["stats"].get("batting", {})
            order = int(p["battingOrder"])
            bats.append({**base, "batter_id": p["person"]["id"],
                         # 100, 200, ... 900 = starting lineup slot; 101 etc = sub
                         "slot": order // 100 if order % 100 == 0 else 0,
                         "pa": s.get("plateAppearances", 0), "ab": s.get("atBats", 0),
                         "h": s.get("hits", 0), "2b": s.get("doubles", 0),
                         "3b": s.get("triples", 0), "hr": s.get("homeRuns", 0),
                         "bb": s.get("baseOnBalls", 0), "k": s.get("strikeOuts", 0),
                         "hbp": s.get("hitByPitch", 0), "sf": s.get("sacFlies", 0)})
    return starts, bats


def pull_season(season):
    games = schedule(season)
    starts, bats = [], []
    with ThreadPoolExecutor(8) as pool:
        for i, (s, b) in enumerate(pool.map(lambda kv: parse_game(*kv), games.items())):
            starts += s
            bats += b
            if i % 500 == 0:
                print(season, i, "/", len(games), flush=True)
    pd.DataFrame(starts).assign(season=season).to_csv(RAW / f"starts_{season}.csv.gz", index=False)
    pd.DataFrame(bats).assign(season=season).to_csv(RAW / f"batting_{season}.csv.gz", index=False)

In [3]:
# skips seasons already saved; takes ~3 min per season
for season in SEASONS:
    if not (RAW / f"starts_{season}.csv.gz").exists():
        pull_season(season)

2021 0 / 2429
2021 500 / 2429
2021 1000 / 2429
2021 1500 / 2429
2021 2000 / 2429
2022 0 / 2430
2022 500 / 2430
2022 1000 / 2430
2022 1500 / 2430
2022 2000 / 2430
2023 0 / 2430
2023 500 / 2430
2023 1000 / 2430
2023 1500 / 2430
2023 2000 / 2430
2024 0 / 2429
2024 500 / 2429
2024 1000 / 2429
2024 1500 / 2429
2024 2000 / 2429
2025 0 / 2430
2025 500 / 2430
2025 1000 / 2430
2025 1500 / 2430
2025 2000 / 2430
2026 0 / 2429
2026 500 / 2429
2026 1000 / 2429
2026 1500 / 2429
2026 2000 / 2429


In [4]:
# MLB Stats API play-by-play associates each completed plate appearance with
# matchup.batter.id and matchup.pitcher.id. We retain only official ABs.
# These outcomes are not official at-bats under MLB scoring rules.
NON_AB_EVENTS = {
    "walk", "intent_walk", "intentional_walk", "hit_by_pitch",
    "sac_fly", "sac_bunt", "sac_fly_double_play", "sac_bunt_double_play",
    "catcher_interf", "catcher_interference", "fan_interference",
}
HIT_EVENTS = {"single", "double", "triple", "home_run"}


def parse_bvp_game(game_pk, date):
    data = get(f"{API}/game/{game_pk}/playByPlay")
    rows = []
    for play in data.get("allPlays", []):
        result = play.get("result", {})
        if result.get("type") != "atBat" or not play.get("about", {}).get("isComplete", False):
            continue
        event = result.get("eventType")
        if event in NON_AB_EVENTS:
            continue
        # An unknown/missing outcome should never be silently treated as an AB.
        if not event:
            raise ValueError(f"Missing eventType in game {game_pk}, play {play.get('about', {})}")
        matchup = play.get("matchup", {})
        batter_id = matchup.get("batter", {}).get("id")
        pitcher_id = matchup.get("pitcher", {}).get("id")
        if not batter_id or not pitcher_id:
            raise ValueError(f"Missing matchup IDs in game {game_pk}")
        rows.append({"game_pk": game_pk, "date": date,
                     "batter_id": batter_id, "pitcher_id": pitcher_id,
                     "ab": 1, "h": int(event in HIT_EVENTS)})
    return rows


def pull_bvp_season(season):
    games = schedule(season)
    rows = []
    # Reuse the schedule and retry logic defined in the original notebook.
    with ThreadPoolExecutor(max_workers=8) as pool:
        for i, game_rows in enumerate(pool.map(lambda kv: parse_bvp_game(*kv), games.items())):
            rows.extend(game_rows)
            if (i + 1) % 500 == 0:
                print("BvP", season, i + 1, "/", len(games), flush=True)
    if not rows:
        raise ValueError(f"No BvP at-bats found for season {season}")
    path = RAW / f"bvp_{season}.csv.gz"
    pd.DataFrame(rows).assign(season=season).to_csv(path, index=False)
    print("Saved", path, "with", len(rows), "at-bats")


# Check both source files; skip a season only when its BvP cache exists.
for season in SEASONS:
    if not (RAW / f"bvp_{season}.csv.gz").exists():
        pull_bvp_season(season)

BvP 2021 500 / 2429
BvP 2021 1000 / 2429
BvP 2021 1500 / 2429
BvP 2021 2000 / 2429
Saved data/raw/bvp_2021.csv.gz with 162282 at-bats
BvP 2022 500 / 2430
BvP 2022 1000 / 2430
BvP 2022 1500 / 2430
BvP 2022 2000 / 2430
Saved data/raw/bvp_2022.csv.gz with 163879 at-bats
BvP 2023 500 / 2430
BvP 2023 1000 / 2430
BvP 2023 1500 / 2430
BvP 2023 2000 / 2430
Saved data/raw/bvp_2023.csv.gz with 164820 at-bats
BvP 2024 500 / 2429
BvP 2024 1000 / 2429
BvP 2024 1500 / 2429
BvP 2024 2000 / 2429
Saved data/raw/bvp_2024.csv.gz with 164141 at-bats
BvP 2025 500 / 2430
BvP 2025 1000 / 2430
BvP 2025 1500 / 2430
BvP 2025 2000 / 2430
Saved data/raw/bvp_2025.csv.gz with 164125 at-bats
BvP 2026 500 / 2429
BvP 2026 1000 / 2429
BvP 2026 1500 / 2429
BvP 2026 2000 / 2429
Saved data/raw/bvp_2026.csv.gz with 163809 at-bats


In [5]:
def pull_people():
    ids = set()
    for f in RAW.glob("starts_*.csv.gz"):
        ids |= set(pd.read_csv(f)["pitcher_id"])
    for f in RAW.glob("batting_*.csv.gz"):
        ids |= set(pd.read_csv(f)["batter_id"])
    ids = sorted(ids)
    rows = []
    for i in range(0, len(ids), 200):
        chunk = ",".join(map(str, ids[i:i + 200]))
        for p in get(f"{API}/people", {"personIds": chunk})["people"]:
            rows.append({"player_id": p["id"], "name": p["fullName"],
                         "throws": p.get("pitchHand", {}).get("code"),
                         "bats": p.get("batSide", {}).get("code")})
    pd.DataFrame(rows).to_csv(RAW / "players.csv.gz", index=False)


if not (RAW / "players.csv.gz").exists():
    pull_people()

## Build features

In [6]:
SHRINK = 100  # pseudo PA/BF to pull small samples toward league average
LAST_N = 5


def load(prefix):
    df = pd.concat([pd.read_csv(f) for f in sorted(RAW.glob(f"{prefix}_*.csv.gz"))])
    df["date"] = pd.to_datetime(df["date"])
    return df


def shrink(num, den, prior):
    return (num + prior * SHRINK) / (den + SHRINK)


def window_sums(df, key, cols):
    # sums over the previous 365 days, excluding the current date
    daily = df.groupby([key, "date"])[cols].sum().reset_index().sort_values([key, "date"])
    return (daily.set_index("date").groupby(key)[cols]
            .rolling("365D", closed="left").sum().fillna(0).reset_index())

In [7]:
def pitcher_features(starts, throws):
    s = starts.sort_values(["pitcher_id", "date", "game_pk"]).reset_index(drop=True)
    train = s[s.season.isin(TRAIN_SEASONS)]
    per_start = train[["h", "k", "bb", "er", "outs", "pitches"]].mean()
    lg = train[["h", "k", "bb", "er", "hr", "bf", "outs"]].sum()

    g = s.groupby("pitcher_id")
    for c in ["h", "k", "bb", "er", "outs", "pitches"]:
        s[f"p_l{LAST_N}_{c}"] = g[c].transform(
            lambda x: x.shift(1).rolling(LAST_N, min_periods=1).mean()).fillna(per_start[c])
    s["p_rest_days"] = g["date"].diff().dt.days.clip(upper=30).fillna(30)

    s["n"] = 1
    r = window_sums(s, "pitcher_id", ["n", "bf", "outs", "h", "k", "bb", "er", "hr"])
    s = s.merge(r, on=["pitcher_id", "date"], suffixes=("", "_365"))
    s = s.sort_values(["pitcher_id", "date", "game_pk"]).reset_index(drop=True)
    s["p_n_starts"] = s["n_365"]
    for c in ["k", "bb", "h", "hr"]:
        s[f"p_{c}_rate"] = shrink(s[f"{c}_365"], s["bf_365"], lg[c] / lg["bf"])
    s["p_er9"] = 27 * shrink(s["er_365"], s["outs_365"], lg["er"] / lg["outs"])
    s["p_outs_per_start"] = (s["outs_365"] + 3 * per_start["outs"]) / (s["n_365"] + 3)

    # season-to-date per-start averages (for the average baseline)
    gs = s.groupby(["pitcher_id", "season"])
    s["p_std_n"] = gs.cumcount()
    for c in ["h", "k", "bb", "er"]:
        prev = gs[c].cumsum() - s[c]
        s[f"p_std_{c}"] = np.where(s["p_std_n"] > 0, prev / s["p_std_n"].clip(lower=1),
                                   s[f"p_l{LAST_N}_{c}"])

    s = s.merge(throws, left_on="pitcher_id", right_on="player_id", how="left")
    s["p_throws_R"] = (s["throws"] == "R").astype(int)
    return s.drop(columns=[c for c in s.columns if c.endswith("_365")] + ["n", "player_id", "throws"])


def batter_features(bat):
    cols = ["pa", "ab", "h", "2b", "3b", "hr", "bb", "k", "hbp", "sf"]
    t = bat[bat.season.isin(TRAIN_SEASONS)][cols].sum()
    r = window_sums(bat, "batter_id", cols)
    out = r[["batter_id", "date", "pa"]].copy()
    out["k_rate"] = shrink(r["k"], r["pa"], t["k"] / t["pa"])
    out["bb_rate"] = shrink(r["bb"], r["pa"], t["bb"] / t["pa"])
    out["obp"] = shrink(r["h"] + r["bb"] + r["hbp"], r["pa"], (t["h"] + t["bb"] + t["hbp"]) / t["pa"])
    out["iso"] = shrink(r["2b"] + 2 * r["3b"] + 3 * r["hr"], r["ab"],
                        (t["2b"] + 2 * t["3b"] + 3 * t["hr"]) / t["ab"])
    return out

In [8]:
def historical_bvp(lineup_pairs, bvp_atbats):
    """Pregame BvP for each starting-lineup batter vs that day's starter.

    lineup_pairs columns: game_pk, date, pitcher_id, batter_id, slot
    bvp_atbats columns: date, pitcher_id, batter_id, ab, h
    Returns one row per lineup pair; missing average means no prior ABs.
    """
    left = lineup_pairs[["game_pk", "date", "pitcher_id", "batter_id", "slot"]].copy()
    left["date"] = pd.to_datetime(left["date"])
    history = bvp_atbats[["date", "pitcher_id", "batter_id", "ab", "h"]].copy()
    history["date"] = pd.to_datetime(history["date"])
    if not history.empty:
        first = pd.Timestamp("2021-01-01")
        history = history[history["date"] >= first]
    if history.empty:
        left["bvp_h"] = 0
        left["bvp_ab"] = 0
    else:
        daily = (history.groupby(["pitcher_id", "batter_id", "date"], as_index=False)
                 [["ab", "h"]].sum())
        daily = daily.sort_values(["pitcher_id", "batter_id", "date"])
        daily[["bvp_ab", "bvp_h"]] = daily.groupby(["pitcher_id", "batter_id"])[["ab", "h"]].cumsum()
        # pandas merge_asof requires global sorting by the merge date.
        daily = daily[["date", "pitcher_id", "batter_id", "bvp_ab", "bvp_h"]].sort_values("date")
        left = pd.merge_asof(left.sort_values("date"), daily,
                             on="date", by=["pitcher_id", "batter_id"],
                             direction="backward", allow_exact_matches=False)
        left[["bvp_ab", "bvp_h"]] = left[["bvp_ab", "bvp_h"]].fillna(0).astype("int64")
    left["bvp_avg"] = np.where(left["bvp_ab"] > 0,
                                left["bvp_h"] / left["bvp_ab"].replace(0, np.nan),
                                np.nan)
    return left

In [9]:
starts = load("starts")
bat = load("batting")
players = pd.read_csv(RAW / "players.csv.gz")

df = pitcher_features(starts, players[["player_id", "throws"]])

lineup = bat[bat.slot > 0][["game_pk", "team_id", "batter_id", "date", "slot"]]
lineup = lineup.merge(batter_features(bat), on=["batter_id", "date"], how="left")
lineup = lineup.merge(players[["player_id", "bats"]], left_on="batter_id",
                      right_on="player_id", how="left").drop(columns="player_id")

# only keep games with a full 1-9 lineup
full = lineup.groupby(["game_pk", "team_id"])["slot"].nunique() == 9
lineup = lineup.set_index(["game_pk", "team_id"]).loc[full[full].index].reset_index()

df = df.merge(lineup[["game_pk", "team_id", "slot", "batter_id", "bats", "pa", "k_rate",
                      "bb_rate", "obp", "iso"]],
              left_on=["game_pk", "opp_id"], right_on=["game_pk", "team_id"],
              suffixes=("", "_b"))
df["platoon_adv"] = ((df["bats"] == "S") |
                     ((df["bats"] == "L") & (df["p_throws_R"] == 1)) |
                     ((df["bats"] == "R") & (df["p_throws_R"] == 0))).astype(int)
df["bats_L"] = (df["bats"] == "L").astype(int)

In [10]:
# Attach raw historical BvP by batter and opposing starting pitcher.
# The original df has one row per lineup slot for each starting pitcher.
bvp = load("bvp")
bvp_pairs = historical_bvp(
    df[["game_pk", "date", "pitcher_id", "batter_id", "slot"]], bvp
)
assert not bvp_pairs.duplicated(["game_pk", "pitcher_id", "slot"]).any()
df = df.merge(bvp_pairs[["game_pk", "pitcher_id", "slot", "bvp_h", "bvp_ab", "bvp_avg"]],
              on=["game_pk", "pitcher_id", "slot"], how="left", validate="one_to_one")
assert df["bvp_ab"].notna().all()

In [11]:
feats = ["pa", "k_rate", "bb_rate", "obp", "iso", "platoon_adv", "bats_L",
         "bvp_h", "bvp_ab", "bvp_avg"]
key = ["game_pk", "pitcher_id"]

wide = df.pivot(index=key, columns="slot", values=feats)
wide.columns = [f"b{slot}_{f}" for f, slot in wide.columns]

# Original lineup means exclude BvP: an unweighted mean of nine averages is
# NOT a valid aggregate batting average when hitters have different AB counts.
old_feats = ["pa", "k_rate", "bb_rate", "obp", "iso", "platoon_adv", "bats_L"]
agg = df.groupby(key)[old_feats].mean().add_prefix("lu_")

# Combined BvP: sum H / sum AB across ONLY batters with earlier AB against
# this starter. Batters with no history contribute 0 to numerator/denominator.
lu_bvp = df.groupby(key).agg(
    lu_bvp_h=("bvp_h", "sum"),
    lu_bvp_ab=("bvp_ab", "sum"),
    lu_bvp_batters=("bvp_ab", lambda x: int((x > 0).sum())),
)
lu_bvp["lu_bvp_avg"] = lu_bvp["lu_bvp_h"].div(lu_bvp["lu_bvp_ab"].replace(0, np.nan))
ids = df.pivot(index=key, columns="slot", values="batter_id").add_prefix("b").add_suffix("_id")

base = df.drop_duplicates(key).set_index(key)
targets = base[["h", "k", "bb", "er"]].rename(
    columns={"h": "hits", "k": "strikeouts", "bb": "walks", "er": "earned_runs"})
p_cols = [c for c in base.columns if c.startswith("p_")] + ["home"]

dataset = pd.concat([base[["date", "season", "team_id", "opp_id"]], targets,
                     base[p_cols], agg, lu_bvp, wide, ids], axis=1).reset_index()
dataset = dataset.sort_values(["date", "game_pk"]).reset_index(drop=True)
dataset.to_csv(DATA / "starts_dataset.csv.gz", index=False)

print(f"{len(dataset)} starts, {dataset.shape[1]} columns")
dataset.groupby("season").size()

29154 starts, 141 columns


,0
season,
2021,4858
2022,4860
2023,4860
2024,4858
2025,4860
2026,4858


## Checks

In [12]:
# BvP average NaNs are intentional (no previous at-bats vs starter).
non_bvp_avg_cols = [c for c in dataset.columns if not c.endswith("_bvp_avg")]
print("unexpected missing values:", dataset[non_bvp_avg_cols].isna().sum().sum())
print("duplicate starts:", dataset.duplicated(["game_pk", "pitcher_id"]).sum())
print("lineups with no prior BvP at-bats:", (dataset["lu_bvp_ab"] == 0).sum())
assert dataset[non_bvp_avg_cols].notna().all().all()
assert not dataset.duplicated(["game_pk", "pitcher_id"]).any()
assert (dataset["lu_bvp_avg"].isna() == (dataset["lu_bvp_ab"] == 0)).all()
assert (dataset["lu_bvp_h"] <= dataset["lu_bvp_ab"]).all()
assert (dataset["lu_bvp_batters"].between(0, 9)).all()
assert (dataset["lu_bvp_h"] == dataset[[f"b{i}_bvp_h" for i in range(1, 10)]].sum(axis=1)).all()
assert (dataset["lu_bvp_ab"] == dataset[[f"b{i}_bvp_ab" for i in range(1, 10)]].sum(axis=1)).all()
dataset[["hits", "strikeouts", "walks", "earned_runs", "lu_bvp_avg", "lu_bvp_ab"]].describe().round(3)

unexpected missing values: 0
duplicate starts: 0
lineups with no prior BvP at-bats: 6993


,hits,strikeouts,walks,earned_runs,lu_bvp_avg,lu_bvp_ab
count,29154.000,29154.000,29154.000,29154.000,22161.000,29154.00
mean,4.874,4.781,1.698,2.419,0.253,20.54
std,2.256,2.523,1.290,1.977,0.140,23.75
min,0.000,0.000,0.000,0.000,0.000,0.00
25%,3.000,3.000,1.000,1.000,0.178,1.00
50%,5.000,5.000,2.000,2.000,0.250,14.00
75%,6.000,6.000,2.000,4.000,0.323,30.00
max,14.000,16.000,8.000,13.000,1.000,242.00


In [13]:
print("=== STARTS DATAHEAD ===")
display(starts.head(3))

print("\n=== BATTING DATAHEAD ===")
display(bat.head(3))

print("\n=== PLAYERS DATAHEAD ===")
display(players.head(3))

print("\n=== FINAL DATASET HEAD ===")
display(dataset.head(3))

=== STARTS DATAHEAD ===


,game_pk,date,team_id,opp_id,home,pitcher_id,outs,bf,pitches,h,k,bb,er,hr,hbp,season
0,634642,2021-04-01,147,141,1,543037,16,23,97,5,8,2,2,1,0,2021
1,634642,2021-04-01,141,147,0,547943,16,21,92,4,5,1,2,1,0,2021
2,634645,2021-04-01,116,114,1,571510,17,24,92,3,2,4,0,0,0,2021



=== BATTING DATAHEAD ===


,game_pk,date,team_id,opp_id,home,batter_id,slot,pa,ab,h,2b,3b,hr,bb,k,hbp,sf,season
0,634642,2021-04-01,147,141,1,518934,1,5,4,0,0,0,0,1,0,0,0,2021
1,634642,2021-04-01,147,141,1,643565,0,0,0,0,0,0,0,0,0,0,0,2021
2,634642,2021-04-01,147,141,1,640449,9,4,3,2,0,0,0,1,0,0,0,2021



=== PLAYERS DATAHEAD ===


,player_id,name,throws,bats
0,405395,Albert Pujols,R,R
1,408234,Miguel Cabrera,R,R
2,424144,Oliver Pérez,L,L



=== FINAL DATASET HEAD ===


,game_pk,pitcher_id,date,season,team_id,opp_id,hits,strikeouts,walks,earned_runs,...,b9_bvp_avg,b1_id,b2_id,b3_id,b4_id,b5_id,b6_id,b7_id,b8_id,b9_id
0,634615,477132,2021-04-01,2021,119,115,10,2,1,5,...,NaN,606132,658069,596115,453568,543068,553869,572008,641658,608566
1,634615,608566,2021-04-01,2021,115,119,6,2,6,1,...,NaN,605141,608369,457759,641355,571970,621035,666158,605131,477132
2,634618,506433,2021-04-01,2021,135,109,8,6,1,4,...,NaN,668942,606466,572233,444482,452678,500871,656976,519390,518516
